# POC: regime routing + futures long/short + honest evaluation

**Changes vs the old grid:**
- `imbalance` is **not** run standalone — only **combinations** (`momentum_with_imbalance_confirm`, `mean_with_imbalance_confirm`).
- All tactics use a **linear futures-style** simulator: `long` / `short` (not spot-only long).
- **Regime** (`regime_meta_strategy`): EMA trend on 1m bars; per tick — mean-reversion in **range**, momentum + breakout in **trend**.
- **Hold-out:** months in `HOLDOUT_MONTHS` with `eval_label=holdout_oos` — **do not tune parameters to these** in the same run. Explore on train first, then frozen params on OOS.
- **TP/SL:** sections **A / B / C** run **twice**: first **fixed** (`adaptive_tp_sl=False`), then the **same strategy grid** with **adaptive** TP/SL at entry (`_FIXED_KW` / `_ADAPT_KW` in the config cell; see `ADAPTIVE_VOL_*` in `poc_backtest_core.py`).

See also: `POC_METHODOLOGY.md`, trend code: `poc_trend_detection.py`, core: `poc_backtest_core.py`.

In [1]:
import os

from poc_backtest_core import (
    TP_SL_PRESETS,
    IMBALANCE_THRESHOLDS,
    MEAN_DELTAS,
    MOMENTUM_THRESHOLDS,
    BREAKOUT_LOOKBACKS,
    analytics_full,
    clear_reports,
    mean_strategy_futures,
    momentum_strategy_futures,
    breakout_strategy_futures,
    momentum_with_imbalance_confirm,
    mean_with_imbalance_confirm,
    regime_meta_strategy,
    run_backtest_yearly,
    ADAPTIVE_VOL_WINDOW,
    ADAPTIVE_VOL_MULT_MIN,
    ADAPTIVE_VOL_MULT_MAX,
)

data_folder = "data"
symbol = "BTCUSDT"
year = 2025

# Full year — exploratory only (you may tune hyperparameters on TRAIN_MONTHS manually)
months_full = list(range(1, 13))
# Hold-out months for the honest leaderboard row (default Q4; change for your test design)
HOLDOUT_MONTHS = (10, 11, 12)
TRAIN_MONTHS = tuple(m for m in range(1, 13) if m not in HOLDOUT_MONTHS)

# Two-phase TP/SL: fixed baseline, then same strategies with volatility-scaled TP/SL at entry.
_FIXED_KW = dict(
    adaptive_tp_sl=False,
    vol_window=ADAPTIVE_VOL_WINDOW,
    vol_mult_min=ADAPTIVE_VOL_MULT_MIN,
    vol_mult_max=ADAPTIVE_VOL_MULT_MAX,
)
_ADAPT_KW = dict(
    adaptive_tp_sl=True,
    vol_window=ADAPTIVE_VOL_WINDOW,
    vol_mult_min=ADAPTIVE_VOL_MULT_MIN,
    vol_mult_max=ADAPTIVE_VOL_MULT_MAX,
)


def run_full_year_grid(_tp_sl_kw):
    for _preset, (_tp, _sl) in TP_SL_PRESETS.items():
        for _thr in IMBALANCE_THRESHOLDS:
            run_backtest_yearly(
                "mom+imb",
                momentum_with_imbalance_confirm,
                data_folder=data_folder,
                symbol=symbol,
                year=year,
                months=months_full,
                eval_label="full_year",
                threshold=_thr,
                momentum_threshold=0.001,
                tp=_tp,
                sl=_sl,
                **_tp_sl_kw,
            )
            run_backtest_yearly(
                "mean+imb",
                mean_with_imbalance_confirm,
                data_folder=data_folder,
                symbol=symbol,
                year=year,
                months=months_full,
                eval_label="full_year",
                threshold=_thr,
                delta=0.002,
                tp=_tp,
                sl=_sl,
                **_tp_sl_kw,
            )

    for _preset, (_tp, _sl) in TP_SL_PRESETS.items():
        for _delta in MEAN_DELTAS:
            run_backtest_yearly(
                "mean",
                mean_strategy_futures,
                data_folder=data_folder,
                symbol=symbol,
                year=year,
                months=months_full,
                eval_label="full_year",
                delta=_delta,
                tp=_tp,
                sl=_sl,
                **_tp_sl_kw,
            )
        for _mt in MOMENTUM_THRESHOLDS:
            run_backtest_yearly(
                "momentum",
                momentum_strategy_futures,
                data_folder=data_folder,
                symbol=symbol,
                year=year,
                months=months_full,
                eval_label="full_year",
                momentum_threshold=_mt,
                tp=_tp,
                sl=_sl,
                **_tp_sl_kw,
            )
        for _lb in BREAKOUT_LOOKBACKS:
            run_backtest_yearly(
                "breakout",
                breakout_strategy_futures,
                data_folder=data_folder,
                symbol=symbol,
                year=year,
                months=months_full,
                eval_label="full_year",
                lookback=_lb,
                tp=_tp,
                sl=_sl,
                **_tp_sl_kw,
            )
        run_backtest_yearly(
            "regime_meta",
            regime_meta_strategy,
            data_folder=data_folder,
            symbol=symbol,
            year=year,
            months=months_full,
            eval_label="full_year",
            mean_delta=0.002,
            mom_thr=0.001,
            br_lookback=500,
            tp=_tp,
            sl=_sl,
            **_tp_sl_kw,
        )


def run_holdout_grid(_tp_sl_kw):
    for _preset, (_tp, _sl) in TP_SL_PRESETS.items():
        for _thr in IMBALANCE_THRESHOLDS:
            run_backtest_yearly(
                "mom+imb",
                momentum_with_imbalance_confirm,
                data_folder=data_folder,
                symbol=symbol,
                year=year,
                months=HOLDOUT_MONTHS,
                eval_label="holdout_oos",
                threshold=_thr,
                momentum_threshold=0.001,
                tp=_tp,
                sl=_sl,
                **_tp_sl_kw,
            )
        for _delta in MEAN_DELTAS:
            run_backtest_yearly(
                "mean",
                mean_strategy_futures,
                data_folder=data_folder,
                symbol=symbol,
                year=year,
                months=HOLDOUT_MONTHS,
                eval_label="holdout_oos",
                delta=_delta,
                tp=_tp,
                sl=_sl,
                **_tp_sl_kw,
            )
        run_backtest_yearly(
            "regime_meta",
            regime_meta_strategy,
            data_folder=data_folder,
            symbol=symbol,
            year=year,
            months=HOLDOUT_MONTHS,
            eval_label="holdout_oos",
            mean_delta=0.002,
            mom_thr=0.001,
            br_lookback=500,
            tp=_tp,
            sl=_sl,
            **_tp_sl_kw,
        )


def run_forward_oos_grid(oos_year, oos_months, eval_label, _tp_sl_kw):
    for _preset, (_tp, _sl) in TP_SL_PRESETS.items():
        for _thr in IMBALANCE_THRESHOLDS:
            run_backtest_yearly(
                "mom+imb",
                momentum_with_imbalance_confirm,
                data_folder=data_folder,
                symbol=symbol,
                year=oos_year,
                months=oos_months,
                eval_label=eval_label,
                threshold=_thr,
                momentum_threshold=0.001,
                tp=_tp,
                sl=_sl,
                **_tp_sl_kw,
            )
            run_backtest_yearly(
                "mean+imb",
                mean_with_imbalance_confirm,
                data_folder=data_folder,
                symbol=symbol,
                year=oos_year,
                months=oos_months,
                eval_label=eval_label,
                threshold=_thr,
                delta=0.002,
                tp=_tp,
                sl=_sl,
                **_tp_sl_kw,
            )
        for _delta in MEAN_DELTAS:
            run_backtest_yearly(
                "mean",
                mean_strategy_futures,
                data_folder=data_folder,
                symbol=symbol,
                year=oos_year,
                months=oos_months,
                eval_label=eval_label,
                delta=_delta,
                tp=_tp,
                sl=_sl,
                **_tp_sl_kw,
            )
        run_backtest_yearly(
            "regime_meta",
            regime_meta_strategy,
            data_folder=data_folder,
            symbol=symbol,
            year=oos_year,
            months=oos_months,
            eval_label=eval_label,
            mean_delta=0.002,
            mom_thr=0.001,
            br_lookback=500,
            tp=_tp,
            sl=_sl,
            **_tp_sl_kw,
        )


TP_SL_PRESETS = {
    "tight": (0.003, 0.002),
    "balanced": (0.007, 0.004),
    "swing": (0.012, 0.006),
}

MEAN_DELTAS = [0.002, 0.003]
MOMENTUM_THRESHOLDS = [0.001, 0.002]
BREAKOUT_LOOKBACKS = [500, 800]
IMBALANCE_THRESHOLDS = [1.25]  # for combined filters only

_sample = os.path.join(data_folder, f"{symbol}-aggTrades-{year}-01.parquet")
if not os.path.isfile(_sample):
    print(f"No data at {_sample!r} — place parquet files under ./data/")

## A) Exploratory: full year (all months in sequence, one account)
Use for **comparing ideas**, not as final proof of edge. Runs **fixed TP/SL first**, then **adaptive** (same presets).

In [2]:
clear_reports()

if os.path.isfile(_sample):
    print("\n=== A) full_year — fixed TP/SL ===\n")
    run_full_year_grid(_FIXED_KW)
    print("\n=== A) full_year — adaptive TP/SL ===\n")
    run_full_year_grid(_ADAPT_KW)

    analytics_full("poc_backtest_leaderboard_full_year.csv")


=== A) full_year — fixed TP/SL ===

Running [full_year] mom+imb {'threshold': 1.25, 'momentum_threshold': 0.001, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 01
Running [full_year] mom+imb {'threshold': 1.25, 'momentum_threshold': 0.001, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 02
Running [full_year] mom+imb {'threshold': 1.25, 'momentum_threshold': 0.001, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 03
Running [full_year] mom+imb {'threshold': 1.25, 'momentum_threshold': 0.001, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 04
Running [full_year] mom+imb {'threshold': 1.25, 'momentum_threshold': 0.001, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 05
Running 

## B) Hold-out OOS: only months in `HOLDOUT_MONTHS`
Separate account from `INITIAL_BALANCE`, **no** compounding from train. Parameters should be **frozen** after analysis on train. **Fixed** TP/SL run first, then **adaptive**.

In [3]:
clear_reports()

if os.path.isfile(_sample):
    print("\n=== B) holdout_oos — fixed TP/SL ===\n")
    run_holdout_grid(_FIXED_KW)
    print("\n=== B) holdout_oos — adaptive TP/SL ===\n")
    run_holdout_grid(_ADAPT_KW)

    analytics_full("poc_backtest_leaderboard_holdout_oos.csv")


=== B) holdout_oos — fixed TP/SL ===

Running [holdout_oos] mom+imb {'threshold': 1.25, 'momentum_threshold': 0.001, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 10
Running [holdout_oos] mom+imb {'threshold': 1.25, 'momentum_threshold': 0.001, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 11
Running [holdout_oos] mom+imb {'threshold': 1.25, 'momentum_threshold': 0.001, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 12
Running [holdout_oos] mean {'delta': 0.002, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 10
Running [holdout_oos] mean {'delta': 0.002, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 11
Running [holdout_oos] mean {'delta': 0.002, 'adaptive_tp_sl': Fals

## C) Forward OOS — next calendar year (e.g. 2026)

After 2025 train / hold-out, run the **same frozen parameters** on **new** parquet files you did not tune on. Expected paths: `./data/{symbol}-aggTrades-2026-MM.parquet`. Set `FORWARD_OOS_MONTHS` to the months you actually downloaded.

This is an extra **sanity check** on unseen data, not a full walk-forward study. Output CSV: `poc_backtest_leaderboard_forward_oos_<year>.csv`. **Fixed** TP/SL first, then **adaptive** (same grid).

In [4]:
# Forward out-of-sample: new year, months present on disk (same naming as preprocess)
FORWARD_OOS_YEAR = 2026
FORWARD_OOS_MONTHS = (1, 2, 3)  # edit to match your downloaded parquet months

_forward_sample = os.path.join(
    data_folder, f"{symbol}-aggTrades-{FORWARD_OOS_YEAR}-{FORWARD_OOS_MONTHS[0]:02d}.parquet"
)

clear_reports()

if not os.path.isfile(_forward_sample):
    print(
        f"Skip forward OOS: missing {_forward_sample!r} — place files "
        f"{symbol}-aggTrades-{FORWARD_OOS_YEAR}-MM.parquet for months {FORWARD_OOS_MONTHS} under ./{data_folder}/"
    )
else:
    _eval = f"forward_oos_{FORWARD_OOS_YEAR}"
    _csv_out = f"poc_backtest_leaderboard_forward_oos_{FORWARD_OOS_YEAR}.csv"

    print(f"\n=== C) {_eval} — fixed TP/SL ===\n")
    run_forward_oos_grid(FORWARD_OOS_YEAR, FORWARD_OOS_MONTHS, _eval, _FIXED_KW)
    print(f"\n=== C) {_eval} — adaptive TP/SL ===\n")
    run_forward_oos_grid(FORWARD_OOS_YEAR, FORWARD_OOS_MONTHS, _eval, _ADAPT_KW)

    analytics_full(_csv_out)
    print(f"\nForward OOS label in table: {_eval!r} — compare return_% vs 2025 hold-out.")


=== C) forward_oos_2026 — fixed TP/SL ===

Running [forward_oos_2026] mom+imb {'threshold': 1.25, 'momentum_threshold': 0.001, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 01
Running [forward_oos_2026] mom+imb {'threshold': 1.25, 'momentum_threshold': 0.001, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 02
Running [forward_oos_2026] mom+imb {'threshold': 1.25, 'momentum_threshold': 0.001, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 03
Running [forward_oos_2026] mean+imb {'threshold': 1.25, 'delta': 0.002, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.002 for 01
Running [forward_oos_2026] mean+imb {'threshold': 1.25, 'delta': 0.002, 'adaptive_tp_sl': False, 'vol_window': 2000, 'vol_mult_min': 0.75, 'vol_mult_max': 1.35} tp=0.003 sl=0.0